# Schnorr Signatures and How They Relate to FROST

## Why This Matters

FROST is built so that the final output still looks like a standard Schnorr signature.
That is important for:

- backward compatibility with ordinary Schnorr verification
- keeping the threshold structure hidden from outside verifiers

So before understanding FROST signing, it helps to understand plain Schnorr clearly.

## Schnorr Signature Generation

Let:

- $s \in \mathbb{Z}_q$ be the secret key
- $Y = g^s \in G$ be the public key
- $m$ be the message

To generate a Schnorr signature:

1. Sample random nonce $k \xleftarrow{\$} \mathbb{Z}_q$
2. Compute commitment

$$
R = g^k
$$

3. Compute challenge

$$
c = H(R, Y, m)
$$

4. Compute response

$$
z = k + s \cdot c
$$

5. Output signature

$$
\sigma = (R, z)
$$

## Schnorr Verification

Given message $m$, public key $Y$, and signature $\sigma = (R, z)$:

1. Recompute

$$
c = H(R, Y, m)
$$

2. Compute

$$
R' = g^z \cdot Y^{-c}
$$

3. Accept if

$$
R \stackrel{?}{=} R'
$$

Equivalent form:

$$
g^z \stackrel{?}{=} R \cdot Y^c
$$

Both checks say the same thing.

## Why Verification Works

If the signer used

$$
z = k + sc,
$$

then

$$
g^z = g^{k+sc} = g^k \cdot (g^s)^c = R \cdot Y^c.
$$

So the verifier can check consistency using only the public key and the signature.

## How This Relates to FROST

FROST does not change the final Schnorr signature format. It changes how the values
inside that signature are produced.

In plain Schnorr:

- one signer knows the full secret key $s$
- one signer samples the nonce $k$
- one signer computes the response $z = k + sc$

In FROST:

- the secret key is split across participants as secret shares $s_i$
- the nonce is built from contributions by multiple participants
- each participant contributes part of the final response
- the group still ends up with one Schnorr-style signature $(R, z)$

## The Main Intuition: Partial Secret Key and Partial Nonce

The two most important values in Schnorr are:

- $s$, the secret key
- $k$, the nonce

FROST distributes both of them.

### Partial secret key

Instead of one participant holding all of $s$, each participant holds a share $s_i$.
You can think of $s_i$ as that participant's **partial private key**.

No single participant has the full signing secret, but a threshold subset can work
together using their shares.

The details of how those shares are created are covered in [03-keygen.ipynb](03-keygen.ipynb).

### Partial nonce

Instead of one participant choosing a single nonce $k$, each participant contributes
nonce material that becomes part of the group's effective nonce.

In FROST, this is more structured than plain Schnorr, because the nonce contribution
is tied to the signing context. The preprocessing details are covered in
[04-preprocessing.ipynb](04-preprocessing.ipynb).

For this notebook, the important idea is just:

- plain Schnorr has one nonce $k$
- FROST has participant-level nonce contributions that combine into the group's signing nonce

## What Carries Over from Schnorr

Even though FROST distributes the work, the high-level Schnorr idea stays the same:

- there is still a commitment $R$
- there is still a challenge $c = H(R, Y, m)$
- there is still a response $z$
- the final signature is still verified like Schnorr

So FROST should be understood as **distributed Schnorr signing**, not as a completely
separate signature format.

## Conceptual Note

Schnorr signatures can be viewed as a $\Sigma$-protocol proof of knowledge of the
discrete logarithm of $Y$, made non-interactive with the Fiat-Shamir transform.

FROST keeps that external structure, while splitting the prover's secret key and
nonce work across multiple participants.